In [1]:
import os
import numpy as np
import cv2
import tensorflow as tf
from sklearn.model_selection import train_test_split
from tensorflow.keras.utils import to_categorical

# Define the paths
data_paths = {
    'Bearing Fault': r'E:\Cutting Tool Paper\Dataset\cutting tool data\new_data\CWT\Bearing Fault Data',
    'Gear Fault': r'E:\Cutting Tool Paper\Dataset\cutting tool data\new_data\CWT\Gear Fault Data',
    'Tool Fault': r'E:\Cutting Tool Paper\Dataset\cutting tool data\new_data\CWT\Tool Fault Data',
    'Normal': r'E:\Cutting Tool Paper\Dataset\cutting tool data\new_data\CWT\Normal Data'
}

# Define a function to load images from a directory
def load_images_from_folder(folder):
    images = []
    for filename in os.listdir(folder):
        img = cv2.imread(os.path.join(folder, filename), cv2.IMREAD_GRAYSCALE)
        if img is not None:
            images.append(img)
    return images

# Load all data
data = []
labels = []
label_map = {'Bearing Fault': 0, 'Gear Fault': 1, 'Tool Fault': 2, 'Normal': 3}

for label, folder in data_paths.items():
    images = load_images_from_folder(folder)
    data.extend(images)
    labels.extend([label_map[label]] * len(images))

# Convert to numpy arrays
data = np.array(data)
labels = np.array(labels)

# Normalize data
data = data / 255.0

# Add channel dimension
data = np.expand_dims(data, axis=-1)

# One-hot encode labels
labels = to_categorical(labels, num_classes=4)

# Split into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(data, labels, test_size=0.2, random_state=42)


In [2]:
import pywt
import numpy as np

# Implement the IWTD function with correct handling of wavelet coefficients
def iwtd_denoise(data, wavelet='db1', level=1, a=0.5, b=0.5):
    # Perform wavelet transform
    coeffs = pywt.wavedec2(data, wavelet, level=level)
    # Adjust threshold using parameters a and b
    threshold = a * np.max(coeffs[-1]) * (1 - b)
    
    # Apply threshold to detail coefficients
    denoised_coeffs = [coeffs[0]]  # Keep the approximation coefficients
    denoised_coeffs += [(pywt.threshold(cH, threshold, mode='soft'),
                         pywt.threshold(cV, threshold, mode='soft'),
                         pywt.threshold(cD, threshold, mode='soft')) for cH, cV, cD in coeffs[1:]]
    
    # Perform inverse wavelet transform
    denoised_data = pywt.waverec2(denoised_coeffs, wavelet)
    return denoised_data

# Apply IWTD denoising to training and testing data
X_train_denoised = np.array([iwtd_denoise(img.squeeze()) for img in X_train])
X_test_denoised = np.array([iwtd_denoise(img.squeeze()) for img in X_test])

# Add channel dimension back to denoised data
X_train_denoised = np.expand_dims(X_train_denoised, axis=-1)
X_test_denoised = np.expand_dims(X_test_denoised, axis=-1)


In [3]:
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Conv2D, MaxPooling2D, Flatten, Dense, Reshape, LSTM, Attention
from tensorflow.keras.optimizers import Adam

# Define the CNN-LSTM-Attention model
def build_model(input_shape):
    inputs = Input(shape=input_shape)

    # Convolutional layers
    x = Conv2D(32, (3, 3), activation='relu', padding='same')(inputs)
    x = MaxPooling2D((2, 2), padding='same')(x)
    x = Conv2D(64, (3, 3), activation='relu', padding='same')(x)
    x = MaxPooling2D((2, 2), padding='same')(x)
    x = Conv2D(128, (3, 3), activation='relu', padding='same')(x)
    x = MaxPooling2D((2, 2), padding='same')(x)

    # Reshape for LSTM layer
    x = Reshape((x.shape[1] * x.shape[2], x.shape[3]))(x)

    # LSTM layer
    x = LSTM(64, return_sequences=True)(x)

    # Attention mechanism
    attention = Attention()([x, x])
    x = Flatten()(attention)

    # Fully connected layers
    x = Dense(256, activation='relu')(x)
    outputs = Dense(4, activation='softmax')(x)

    model = Model(inputs, outputs)
    return model

input_shape = X_train_denoised.shape[1:]
model = build_model(input_shape)

# Compile the model
model.compile(optimizer=Adam(learning_rate=0.001), loss='categorical_crossentropy', metrics=['accuracy'])

# Model summary
model.summary()


NameError: name 'Input' is not defined

: 

In [ ]:
# Train the model
history = model.fit(X_train_denoised, y_train, epochs=50, batch_size=16, validation_data=(X_test_denoised, y_test))


In [ ]:
# Evaluate the model
test_loss, test_accuracy = model.evaluate(X_test_denoised, y_test)
print(f'Test accuracy: {test_accuracy * 100:.2f}%')

# Plot training history
import matplotlib.pyplot as plt

plt.plot(history.history['accuracy'], label='train accuracy')
plt.plot(history.history['val_accuracy'], label='val accuracy')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend()
plt.show()

plt.plot(history.history['loss'], label='train loss')
plt.plot(history.history['val_loss'], label='val loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()
plt.show()
